# -1 TODO

add explainer in 3.

# 0. setup

here, we do the setup showcased in 02_eda

In [1]:
%pip install -r requirements_oldpy.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements_oldpy.txt'

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from scipy import stats
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from pycaret.regression import *

Connect to database

In [3]:
load_dotenv()   # finds the .env in the repo root, also from citibike/notebooks/

engine = create_engine(
    "postgresql+psycopg://",
    connect_args={
        "host": "100.88.194.89",
        "port": 5432,
        "dbname": "citibike",
        "user": os.environ["CITIBIKE_DB_USER"],
        "password": os.environ["CITIBIKE_DB_PASSWORD"].strip(),
        "connect_timeout": 10,
    },
)

with engine.connect() as conn:
    print("Connected as", conn.execute(text("SELECT current_user")).scalar())



Connected as r0957896


Two helpers used throughout:

- `sql()` runs a query on the server and returns a DataFrame, and prints how long it took, so we see which
  questions are expensive.
- A small chart style: thin lines, a light grid, and fixed colours per entity - **member is always blue,
  casual always orange; classic bike always green, e-bike always violet** - so a colour means the same thing
  in every chart.

In [4]:
def fmt(value, spec: str = ",") -> str:
    """Format a number with a dot as thousands separator and a comma as decimal separator: 212.310.903 / 0,057."""
    return f"{value:{spec}}".translate(str.maketrans(",.", ".,"))


def sql(query: str, **params) -> pd.DataFrame:
    t0 = time.time()
    df = pd.read_sql(text(query), engine, params=params)
    print(f"{fmt(len(df))} rows in {fmt(time.time() - t0, '.1f')} s")
    return df

COLORS = {"member": "#2a78d6", "casual": "#eb6834", "classic_bike": "#1baf7a", "electric_bike": "#4a3aa7"}
LABELS = {"member": "Member", "casual": "Casual", "classic_bike": "Classic bike", "electric_bike": "E-bike"}

plt.rcParams.update({
    "figure.figsize": (11, 4.2), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.edgecolor": "#8a8986", "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "lines.linewidth": 2, "legend.frameon": False,
})
millions = FuncFormatter(lambda v, _: f"{fmt(v / 1e6, '.1f')} M")
thousands = FuncFormatter(lambda v, _: f"{v / 1e3:.0f} k")

# Every DataFrame the notebook shows gets the same number format as fmt(). Integer columns get a thousands
# separator; float columns get as many decimals as their values need (at most pandas' display precision).
# Columns that hold a year are left alone, otherwise 2025 would show as 2.025.
NO_SEPARATOR = {"year"}

def _decimals(col: pd.Series) -> int:
    values = col.dropna().to_numpy()
    for d in range(pd.get_option("display.precision")):
        if np.allclose(values, values.round(d)):
            return d
    return pd.get_option("display.precision")

def _formatted_html(df: pd.DataFrame) -> str:
    formats = {}
    for name, col in df.items():
        if name in NO_SEPARATOR or pd.api.types.is_bool_dtype(col):
            continue
        if pd.api.types.is_integer_dtype(col):
            formats[name] = fmt
        elif pd.api.types.is_float_dtype(col):
            formats[name] = lambda v, spec=f",.{_decimals(col)}f": fmt(v, spec)
    return df.style.format(formats, na_rep="").to_html(table_attributes='class="dataframe"')

pd.set_option("styler.render.max_rows", 60)   # a styled table does not truncate itself like a plain one
get_ipython().display_formatter.formatters["text/html"].for_type(pd.DataFrame, _formatted_html)

# 1. Trips Pycaret

here, we use the data preperation method discussed in 04_VB_Data_preperations, which is made to predict the values for the daily trips for the whole city

first we use the data setup created before

In [5]:
trips = sql("""
    SELECT

        -- Season
        CASE
            WHEN EXTRACT(MONTH FROM started_at) IN (12, 1, 2)
                THEN 'Winter'
            WHEN EXTRACT(MONTH FROM started_at) IN (3, 4, 5)
                THEN 'Spring'
            WHEN EXTRACT(MONTH FROM started_at) IN (6, 7, 8)
                THEN 'Summer'
            ELSE 'Autumn'
        END AS start_season,

        -- Month
        EXTRACT(MONTH FROM started_at) AS start_month,

        -- Day of the week
        EXTRACT(DOW FROM started_at) AS start_day_of_week,

        -- Time of day
        CASE
            WHEN EXTRACT(HOUR FROM started_at) >= 5
                 AND EXTRACT(HOUR FROM started_at) < 12
                THEN 'Morning'
            WHEN EXTRACT(HOUR FROM started_at) >= 12
                 AND EXTRACT(HOUR FROM started_at) < 17
                THEN 'Afternoon'
            WHEN EXTRACT(HOUR FROM started_at) >= 17
                 AND EXTRACT(HOUR FROM started_at) < 21
                THEN 'Evening'
            ELSE 'Night'
        END AS start_time_of_day,

        start_station_name,
        rideable_type,
        member_casual,

        ROUND(duration_s / 60.0) AS trip_duration_min

    FROM citibike.trips
    WHERE
        duration_s > 0
        AND duration_s <= 7200
        AND started_at IS NOT NULL
        AND start_station_id IS NOT NULL
        AND rideable_type IS NOT NULL
        AND member_casual IS NOT NULL
    LIMIT 50000
""")

trips.head()

50.000 rows in 1,5 s


,start_season,start_month,start_day_of_week,start_time_of_day,start_station_name,rideable_type,member_casual,trip_duration_min
0,Winter,1,1,Afternoon,University Pl & E 8 St,classic_bike,member,6
1,Winter,1,5,Evening,University Pl & E 8 St,classic_bike,casual,5
2,Winter,1,3,Night,Pershing Square North,electric_bike,member,8
3,Winter,1,3,Morning,University Pl & E 8 St,electric_bike,casual,8
4,Winter,1,5,Evening,University Pl & E 8 St,classic_bike,member,11


# 2. Data setup

here, we set up the data to be used by pycaret, we pass it important variables like
- the frame to use for testing
- what is the target value we're predicting
- session : a kind of seed for the data split
- train_size : how we split the data between training & testing data

In [6]:
s = setup(trips, target = 'trip_duration_min', session_id = 123, train_size=0.7)

,Description,Value
0,Session id,123
1,Target,trip_duration_min
2,Target type,Regression
3,Original data shape,"(50000, 8)"
4,Transformed data shape,"(50000, 11)"
5,Transformed train set shape,"(35000, 11)"
6,Transformed test set shape,"(15000, 11)"
7,Numeric features,2
8,Categorical features,5
9,Preprocess,True


# 3. compare

here, we actually test the different regression based models against eachother, to see which model is best,

it decides which of them is best based on 
- MAE
- MSE
- RMSE
- R2
- RMSLE
- MAPE
- TT (sec)

from this, in our tests, we can see that GradientBoostingRegressor is best for our purposes

In [7]:
best = compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,7.6162,116.7034,10.7996,0.0785,0.7172,1.0737,0.2000
ridge,Ridge Regression,7.6495,117.4966,10.8360,0.0724,0.7196,1.0761,0.0350
lar,Least Angle Regression,7.6495,117.4966,10.8360,0.0724,0.7196,1.0761,0.0330
lr,Linear Regression,7.6498,117.4981,10.8361,0.0723,0.7196,1.0762,0.5110
br,Bayesian Ridge,7.6495,117.4986,10.8361,0.0723,0.7197,1.0763,0.0370
en,Elastic Net,7.7492,119.8826,10.9453,0.0536,0.7297,1.1110,0.0370
lasso,Lasso Regression,7.7488,119.9046,10.9463,0.0534,0.7296,1.1105,0.4510
llar,Lasso Least Angle Regression,7.7488,119.9046,10.9463,0.0534,0.7296,1.1105,0.0350
omp,Orthogonal Matching Pursuit,7.7430,120.3096,10.9649,0.0501,0.7290,1.1018,0.0330
huber,Huber Regressor,7.2518,122.0763,11.0450,0.0363,0.6866,0.8690,0.0870


In [8]:
print(best)
evaluate_model(best)

GradientBoostingRegressor(random_state=123)


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

# 4. testing frame with endstation

as a bit of a thought exercise, & to compare, we tried adding endstation value to our model, this is a value we normally wouldnt have access to untill our prediction is finished, but it is a good way to showcase how this column would be of high value to us if it was usable

first like before we start by loading the data now with a new sql statement

In [9]:


tripsEndStation = sql("""
    SELECT

        -- Season
        CASE
            WHEN EXTRACT(MONTH FROM started_at) IN (12, 1, 2)
                THEN 'Winter'
            WHEN EXTRACT(MONTH FROM started_at) IN (3, 4, 5)
                THEN 'Spring'
            WHEN EXTRACT(MONTH FROM started_at) IN (6, 7, 8)
                THEN 'Summer'
            ELSE 'Autumn'
        END AS start_season,

        -- Month
        EXTRACT(MONTH FROM started_at) AS start_month,

        -- Day of the week
        EXTRACT(DOW FROM started_at) AS start_day_of_week,

        -- Time of day
        CASE
            WHEN EXTRACT(HOUR FROM started_at) >= 5
                 AND EXTRACT(HOUR FROM started_at) < 12
                THEN 'Morning'
            WHEN EXTRACT(HOUR FROM started_at) >= 12
                 AND EXTRACT(HOUR FROM started_at) < 17
                THEN 'Afternoon'
            WHEN EXTRACT(HOUR FROM started_at) >= 17
                 AND EXTRACT(HOUR FROM started_at) < 21
                THEN 'Evening'
            ELSE 'Night'
        END AS start_time_of_day,

        start_station_name,
        end_station_name,
        rideable_type,
        member_casual,

        ROUND(duration_s / 60.0) AS trip_duration_min

    FROM citibike.trips
    WHERE
        duration_s > 0
        AND duration_s <= 7200
        AND started_at IS NOT NULL
        AND start_station_id IS NOT NULL
        AND rideable_type IS NOT NULL
        AND member_casual IS NOT NULL
    LIMIT 50000
""")

tripsEndStation.head()

50.000 rows in 1,1 s


,start_season,start_month,start_day_of_week,start_time_of_day,start_station_name,end_station_name,rideable_type,member_casual,trip_duration_min
0,Winter,1,2,Evening,Rivington St & Ridge St,Allen St & Rivington St,electric_bike,member,2
1,Winter,1,5,Morning,Clark St & Henry St,Columbia Heights & Cranberry St,classic_bike,member,2
2,Winter,1,6,Morning,Clark St & Henry St,Columbia Heights & Cranberry St,classic_bike,member,2
3,Winter,1,0,Morning,Clark St & Henry St,Columbia Heights & Cranberry St,classic_bike,member,2
4,Winter,1,6,Morning,Clark St & Henry St,Columbia Heights & Cranberry St,classic_bike,member,2


#

# 5. setup regression tests & compare

here we do the same as before, except now we're using the new dataframe

In [10]:
s = setup(tripsEndStation, target = 'trip_duration_min', session_id = 123, train_size=0.7)

,Description,Value
0,Session id,123
1,Target,trip_duration_min
2,Target type,Regression
3,Original data shape,"(50000, 9)"
4,Transformed data shape,"(50000, 12)"
5,Transformed train set shape,"(35000, 12)"
6,Transformed test set shape,"(15000, 12)"
7,Numeric features,2
8,Categorical features,6
9,Rows with missing values,0.1%


In [11]:
best = compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,6.8433,111.9318,10.5771,0.1451,0.6514,0.8713,0.8860
gbr,Gradient Boosting Regressor,7.6422,118.7419,10.8937,0.0932,0.7127,1.0556,0.2720
lr,Linear Regression,7.7193,120.1576,10.9588,0.0823,0.7194,1.0681,0.0790
ridge,Ridge Regression,7.7193,120.1574,10.9587,0.0823,0.7194,1.0681,0.0660
lar,Least Angle Regression,7.7193,120.1575,10.9588,0.0823,0.7194,1.0681,0.0690
br,Bayesian Ridge,7.7192,120.1552,10.9586,0.0823,0.7194,1.0683,0.0730
en,Elastic Net,7.8306,122.7342,11.0755,0.0627,0.7302,1.1073,0.0670
llar,Lasso Least Angle Regression,7.8341,122.8164,11.0792,0.0621,0.7305,1.1080,0.0670
lasso,Lasso Regression,7.8341,122.8164,11.0792,0.0621,0.7305,1.1080,0.0760
huber,Huber Regressor,7.3422,124.3321,11.1470,0.0507,0.6878,0.8716,0.1980


In [ ]:
print(best)
evaluate_model(best)

RandomForestRegressor(n_jobs=-1, random_state=123)


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…